In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from data_utils.load_data import load_full_volume
from scipy.ndimage import gaussian_filter

In [ ]:
# image_input = '/run/media/anokhver/Data/Veronika/ctu/Microscopy/Microscopy/20251030/2_5_BAEO_1_Multichannel Z-Stack_20251030_96.vsi'
image_input = '/run/media/anokhver/Data/Veronika/ctu/Microscopy/Microscopy/20251030/2_3_PSYHARMIN_2_Multichannel Z-Stack_20251030_67.vsi'
data = load_full_volume(image_input)


### Plotting

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap


def _black_to(rgb):
    """Create a colormap that goes from black to the given RGB color."""
    return LinearSegmentedColormap.from_list("", [(0, 0, 0), rgb])

# Default channel colormaps: black-to-red, black-to-green, black-to-blue
DEFAULT_CHANNEL_CMAPS = [_black_to((1, 0, 0)), _black_to((0, 1, 0)), _black_to((0, 0, 1))]


def show_imgs(images, titles=None, cmap="viridis", figsize=None,
              max_cols=3, split_channels=False, channel_cmaps=None):
    """
    Display one or more images in a grid.

    Args:
        images:          Single image (2D/3D array) or list of images.
        titles:          Optional list of titles, one per image.
        cmap:            Colormap for single-channel images (default: "viridis").
        figsize:         Figure size; auto-scaled from grid if None.
        max_cols:        Max columns in the grid.
        split_channels:  If True, display each channel of a (C,H,W) image
                         as a separate panel with its own colormap.
        channel_cmaps:   List of colormaps for split channels.
    """
    # --- Normalize input to a list of arrays ---
    if isinstance(images, np.ndarray) and images.ndim in (2, 3):
        images = [images]
    else:
        images = list(images)
    if not images:
        raise ValueError("images must contain at least one item")

    if channel_cmaps is None:
        channel_cmaps = DEFAULT_CHANNEL_CMAPS

    # --- Optionally split (C,H,W) images into per-channel panels ---
    # Each panel gets its own colormap (R/G/B); 2D images pass through unchanged.
    per_image_cmaps = None
    if split_channels:
        split_imgs, split_titles, split_cmaps = [], [], []
        for idx, img in enumerate(images):
            img = np.asarray(img)
            base = titles[idx] if titles and idx < len(titles) else None
            if img.ndim == 3:                          # (C, H, W)
                for c in range(img.shape[0]):
                    split_imgs.append(img[c])
                    split_titles.append(f"{base} - ch{c}" if base else None)
                    split_cmaps.append(channel_cmaps[c % len(channel_cmaps)])
            else:                                      # (H, W) — no split needed
                split_imgs.append(img)
                split_titles.append(base)
                split_cmaps.append(cmap)
        images, titles, per_image_cmaps = split_imgs, split_titles, split_cmaps

    # --- Grid layout ---
    n = len(images)
    cols = min(max_cols, n)
    rows = math.ceil(n / cols)
    if figsize is None:
        figsize = (cols * 4, rows * 4)

    fig, axs = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    axs = axs.ravel()

    for i, ax in enumerate(axs):
        if i < n:
            img = np.asarray(images[i])

            # (C,H,W) -> (H,W,C) for matplotlib; detect if it's RGB/RGBA
            is_color = False
            if img.ndim == 3:
                img = np.moveaxis(img, 0, -1)
                is_color = img.shape[-1] in (3, 4)

            # Color images need no colormap; single-channel ones do
            this_cmap = None if is_color else (per_image_cmaps[i] if per_image_cmaps else cmap)
            ax.imshow(img, cmap=this_cmap, vmin=0, vmax=np.percentile(img, 99.5))

            if titles and i < len(titles) and titles[i]:
                ax.set_title(titles[i])
        ax.axis("off")

    plt.tight_layout()
    return fig, axs[:n]

In [ ]:
data_z_sliced  = data[:, 20, :, :]

In [ ]:
show_imgs(data_z_sliced, split_channels=True)

In [ ]:
show_imgs(data_z_sliced)

### Hessian

In [ ]:
def compute_hessian_2d(image, sigma):
    """
    Compute 2D Hessian matrix eigenvalues.

    For a 2D image, Hessian is:
    H = [[Ixx, Ixy],
         [Iyx, Iyy]]

    Returns:
        lambda1, lambda2: eigenvalues sorted by absolute value (|λ1| <= |λ2|)
    """
    # Smooth first (this is the "window" - Gaussian kernel with std=sigma)
    # smoothed = gaussian_filter(image, sigma=sigma)
    smoothed = image
    
    # Compute second derivatives
    Ixx = gaussian_filter(smoothed, sigma=sigma, order=[2, 0])
    Ixy = gaussian_filter(smoothed, sigma=sigma, order=[1, 1])
    Iyy = gaussian_filter(smoothed, sigma=sigma, order=[0, 2])

    # For each pixel, we have a 2x2 Hessian matrix
    # Compute eigenvalues analytically (faster than numpy.linalg.eigh)
    # For 2x2 symmetric matrix: λ = (trace ± sqrt(trace² - 4*det)) / 2
    trace = Ixx + Iyy
    det = Ixx * Iyy - Ixy * Ixy
    discriminant = np.sqrt(np.maximum(trace*trace - 4*det, 0))

    lambda1 = (trace - discriminant) / 2
    lambda2 = (trace + discriminant) / 2

    # Sort by absolute value
    lambda1, lambda2 = np.minimum(np.abs(lambda1), np.abs(lambda2)), \
                       np.maximum(np.abs(lambda1), np.abs(lambda2))

    return lambda1, lambda2

# Frangi 1

In [ ]:
def frangi_filter_2d(lambda1, lambda2, beta=0.5, c=15):
    """
    Frangi vesselness filter for 2D images.

    Parameters:
        beta:
        c:
    
    Multiscale vessel enhancement filtering (MICCAI 1998) 

    """
    # Avoid division by zero
    lambda2_safe = np.where(np.abs(lambda2) < 1e-10, 1e-10, lambda2)

    R_b = lambda1 / lambda2_safe
    S = np.sqrt(lambda1**2 + lambda2**2)

    response = np.zeros_like(lambda1)

    mask = lambda2 > 0

    response[mask]= np.exp(-R_b[mask]**2 / (2*beta**2)) * \
                   (1 - np.exp(-S[mask]**2 / (2*c**2)))

   
    return response

In [ ]:
def multiscale_frangi_2d(image, sigmas=[0.1, 0.3, 0.5, 0.7, 1], beta=0.5):
    """
    Multi-scale Frangi filtering.

    """
    responses = []

    for sigma in sigmas:
        print(f"Processing sigma={sigma}...")
        lambda1, lambda2 = compute_hessian_2d(image, sigma)

        # Adjust c parameter based on sigma (DDeep3M uses c=2e6, but scale it)
        c = 10 * sigma
        response = frangi_filter_2d(lambda1, lambda2, beta, c)
        responses.append(response)

    # Take maximum across scales
    multiscale_response = np.maximum.reduce(responses)

    return multiscale_response, responses

In [ ]:
def show_frangi_mask(original, frangi_response, percentile=90, alpha=0.4):
    """
    Threshold Frangi response into a binary mask and overlay on original.

    Args:
        original:        2D grayscale image (the channel you ran Frangi on)
        frangi_response: Output of frangi_filter_2d or multiscale_frangi_2d
        percentile:      Top N% of Frangi response kept as "structure" (tune this)
        alpha:           Overlay transparency
    """
    # --- Threshold: keep top responses ---
    threshold = np.percentile(frangi_response[frangi_response > 0], percentile)
    mask = frangi_response > threshold

    # --- Display ---
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))

    # 1) Original
    axs[0].imshow(original, cmap="gray", vmin=0, vmax=np.percentile(original, 99.5))
    axs[0].set_title("Original")

    # 2) Binary mask
    axs[1].imshow(mask, cmap="gray")
    axs[1].set_title(f"Mask (top {100-percentile:.0f}%)")

    # 3) Overlay: original in gray, mask in red
    axs[2].imshow(original, cmap="gray", vmin=0, vmax=np.percentile(original, 99.5))
    axs[2].imshow(mask, cmap=_black_to((1, 0, 0)), alpha=alpha, vmin=0, vmax=1)
    axs[2].set_title("Overlay")

    for ax in axs:
        ax.axis("off")
    plt.tight_layout()
    return fig, mask

In [ ]:
sigmas = [0,1, 0.2, 0.5, 1]  
# sigmas = [0.3]  

# enhanced_multiscale, scale_responses = multiscale_frangi_2d(img_normalized[2, :, :], sigmas)
enhanced_multiscale, scale_responses = multiscale_frangi_2d(data_z_sliced[2, :, :], sigmas)


In [ ]:
# Run Frangi on channel 0 (red channel with the dendrites)
channel = 0
img = data_z_sliced[channel]

enhanced, responses = multiscale_frangi_2d(img, sigmas=[0.13])

In [ ]:
fig, mask = show_frangi_mask(img, enhanced, percentile=90)

Create a mask that will get rid of background and highlight the neural structure.

Found original mention in 1998 paper; Hessian matrix is calculated and eighenvalues are extracted.
We calculate how does the intensity changedepeding on the direction.
We see

eigen_1 - 
eigen_2 -
eigen_3 (for 3d) -

The dendrids can be different in size so that's why going thought diff c is done.


### DDeep3D hessian adaptive

In [ ]:
from scipy.ndimage import distance_transform_edt, gaussian_filter
from skimage.filters import threshold_otsu
import numpy as np


# =============================================================================
# Step 1 & 2: Foreground detection + Distance Transform
# =============================================================================

def compute_adaptive_sigma_map(image, fg_threshold=None):
    """
    Binarize the image, compute the Euclidean Distance Transform, and derive
    a per-pixel Gaussian sigma from the DT (DDeep3M+ adaptive window).

    Paper formulas:
        DN = normalize DT to [1, 256]
        R  = log2(DN)   -> window radius in [0, 8]
        sigma = R       (clamped to [1, 8])

    Returns:
        fg_mask:     Binary foreground mask
        dt:          Raw distance transform
        sigma_map:   Per-pixel sigma (float, range [1, 8])
    """
    if fg_threshold is None:
        fg_threshold = threshold_otsu(image)
    fg_mask = image > fg_threshold

    dt = distance_transform_edt(fg_mask)

    # Normalize DT to [1, 256]
    dt_max = dt.max()
    dn = (dt / dt_max * 255 + 1) if dt_max > 0 else np.ones_like(dt)

    # Adaptive radius: R = log2(DN), clamped to [1, 8]
    sigma_map = np.clip(np.log2(np.clip(dn, 1, 256)), 1, 8)

    return fg_mask, dt, sigma_map



In [ ]:

# =============================================================================
# Step 3: Hessian eigenvalues at a single scale
# =============================================================================

def hessian_eigenvalues_2d(image, sigma):
    """
    Compute 2D Hessian eigenvalues at a given sigma with Lindeberg
    scale normalization (sigma^2).

    Returns:
        lam1, lam2: Signed eigenvalues (lam1 <= lam2, by value not abs)
    """
    s2 = sigma ** 2
    Ixx = gaussian_filter(image, sigma=sigma, order=[2, 0]) * s2
    Ixy = gaussian_filter(image, sigma=sigma, order=[1, 1]) * s2
    Iyy = gaussian_filter(image, sigma=sigma, order=[0, 2]) * s2

    trace = Ixx + Iyy
    det = Ixx * Iyy - Ixy ** 2
    discriminant = np.sqrt(np.maximum(trace ** 2 - 4 * det, 0))

    lam1 = (trace - discriminant) / 2  # smaller
    lam2 = (trace + discriminant) / 2  # larger
    return lam1, lam2



In [ ]:

# =============================================================================
# Step 4: Frangi vesselness from eigenvalues
# =============================================================================

def frangi_vesselness_2d(lam1, lam2, beta=0.5, c=2e6):
    """
    2D Frangi vesselness response from signed Hessian eigenvalues.

    For bright tubular structures on a dark background the larger-magnitude
    eigenvalue is negative (concave across the tube).

    Args:
        lam1, lam2:  Signed eigenvalues (lam1 <= lam2)
        beta:        Blob-suppression weight
        c:           Background-suppression weight (scale to intensity range)
    """
    abs1, abs2 = np.abs(lam1), np.abs(lam2)
    sorted_small = np.minimum(abs1, abs2)  # |lambda_1|
    sorted_large = np.maximum(abs1, abs2)  # |lambda_2|

    large_safe = np.where(sorted_large < 1e-10, 1e-10, sorted_large)
    R_b = sorted_small / large_safe
    S = np.sqrt(sorted_small ** 2 + sorted_large ** 2)

    v = np.exp(-R_b ** 2 / (2 * beta ** 2)) * \
        (1 - np.exp(-S ** 2 / (2 * c ** 2)))

    # Zero out where the signed larger-magnitude eigenvalue is positive
    # (not a bright tube)
    signed_large = np.where(abs2 >= abs1, lam2, lam1)
    v[signed_large > 0] = 0

    return v



In [ ]:

# =============================================================================
# Step 5: Soma hole-filling via high DT values
# =============================================================================

def fill_soma_holes(response, dt, percentile=95):
    """
    Frangi turns round soma into rings. Fill them by detecting soma centers
    via high Distance Transform values and setting their response to max.
    """
    if not np.any(dt > 0):
        return response
    threshold = np.percentile(dt[dt > 0], percentile)
    soma_mask = dt > threshold
    response[soma_mask] = response.max()
    return response



In [ ]:

# =============================================================================
# Full pipeline
# =============================================================================

def adaptive_frangi_ddeep3m(image, fg_threshold=None, beta=0.8, c=2e6,
                            soma_percentile=95):
    """
    DDeep3M+ adaptive enhancement filter (2D adaptation).

    Pipeline:
        1. Binarize -> Distance Transform -> per-pixel sigma map
        2. Discretize sigmas to integers (1..8), compute Hessian per scale
        3. Frangi vesselness per scale, assign each pixel its scale's response
        4. Fill soma holes using high-DT regions

    Args:
        image:            2D single-channel array
        fg_threshold:     Foreground threshold (None = Otsu)
        beta:             Frangi blob-suppression
        c:                Frangi background-suppression (2e6 for raw uint16,
                          ~0.5 for [0,1] normalized images)
        soma_percentile:  DT percentile above which pixels are considered soma

    Returns:
        response, fg_mask, sigma_map
    """
    image = image.astype(np.float64)

    # Adaptive sigma from Distance Transform
    fg_mask, dt, sigma_map = compute_adaptive_sigma_map(image, fg_threshold)

    # Discretize sigma -> integer scales
    sigma_discrete = np.round(sigma_map).astype(int)
    unique_sigmas = np.unique(sigma_discrete)

    response = np.zeros_like(image)

    for s in unique_sigmas:
        scale_mask = sigma_discrete == s

        lam1, lam2 = hessian_eigenvalues_2d(image, float(s))
        v = frangi_vesselness_2d(lam1, lam2, beta, c)

        response[scale_mask] = v[scale_mask]

    # Soma hole-filling
    response = fill_soma_holes(response, dt, soma_percentile)

    return response, fg_mask, sigma_map

In [ ]:
channel = 0 # pick the channel with dendrites
img = data_z_sliced[channel].astype(np.float64)

response, fg_mask, sigma_map = adaptive_frangi_ddeep3m(img, soma_percentile=98)

# Visualize the pipeline

In [ ]:

show_imgs(
    [img, fg_mask.astype(np.float32), sigma_map, response],
    titles=["Original", "FG mask (for DT)", "Adaptive sigma map", "Frangi response"],
    cmap="hot",
    max_cols=4,
    figsize=(20, 5)
)


# Show the final binary mask with overlay
fig, final_mask = show_frangi_mask(img, response, percentile=85)